# patch tqdm

> Provides the patching mechanism to intercept tqdm and emit callbacks with that progress information

In [ ]:
#| default_exp patch_tqdm

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from contextlib import contextmanager
from importlib import import_module
import time
from typing import Optional, Callable
from cjm_tqdm_capture.progress_info import ProgressInfo

from itertools import count
_BAR_COUNTER = count(1)

In [ ]:
#| export
def _make_callback_class(
    BaseTqdm,    # Base tqdm class to extend with callback functionality
    default_cb: Optional[Callable[[ProgressInfo], None]],
    min_update_interval: float = 0.1,  # Minimum time between callback invocations (seconds)
    min_delta_pct: float = 1.0,      # emit only if pct moves by >= this
    emit_initial: bool = False       # whether to emit at 0%
):
    "Create a tqdm subclass that emits progress callbacks during iteration"
    class CallbackTqdm(BaseTqdm):
        "Extended tqdm class that invokes callbacks with progress information"
        def __init__(self, *args, **kwargs):
            "Initialize CallbackTqdm with callback configuration"
            user_cb = kwargs.pop('progress_callback', None)
            self._progress_callback = user_cb if user_cb is not None else default_cb
            self._min_update_interval = kwargs.pop('min_update_interval', min_update_interval)
            self._min_delta_pct = kwargs.pop('min_delta_pct', min_delta_pct)
            self._emit_initial = kwargs.pop('emit_initial', emit_initial)
            self._last_cb_time = 0.0
            self._last_pct = -1.0
            self._last_n = -1 # remember last emitted n
            self._done_emitted = False
            self._bar_id = f"bar-{next(_BAR_COUNTER)}"
            super().__init__(*args, **kwargs)
    
        def _emit(
            self,
            force: bool = False  # Bypass throttling and emit callback immediately
        ):
            "Emit progress callback with current state if conditions are met"
            cb = self._progress_callback
            if cb is None:
                return
    
            total = self.total or 0
            pct = (self.n / total * 100.0) if total else 0.0
    
            # skip very first 0% unless requested
            if not force and not self._emit_initial and self.n == 0:
                return
    
            # de-dupe same step; also don't re-emit once done
            if not force:
                if self._done_emitted and pct >= 100.0:
                    return
                if self.n == self._last_n:
                    return
    
                now = time.time()
                if (now - self._last_cb_time) < self._min_update_interval and abs(pct - self._last_pct) < self._min_delta_pct:
                    return
                self._last_cb_time = now
                self._last_pct = pct
    
            fd = getattr(self, 'format_dict', {})
            fmt = getattr(self, 'format_interval', None)
            elapsed   = fmt(fd.get("elapsed"))   if fmt and fd.get("elapsed")   is not None else None
            remaining = fmt(fd.get("remaining")) if fmt and fd.get("remaining") is not None else None
            rate = None
            if fd.get("rate") is not None:
                unit = fd.get("unit", "it")
                rate = f"{fd['rate']:.2f} {unit}/s"
    
            cb(ProgressInfo(
                progress=pct,
                current=self.n,
                total=self.total or None,
                rate=rate,
                elapsed=elapsed,
                remaining=remaining,
                description=self.desc or None,
                raw_output="",
                bar_id=self._bar_id,                    # Unique identifier for this progress bar
                position=getattr(self, 'position', 0),  # Display position for multi-bar scenarios
            ))
    
            self._last_n = self.n # mark last emitted step
            if pct >= 100.0:
                self._done_emitted = True
    
        def display(self, *a, **k):
            "Update display and emit progress callback"
            out = super().display(*a, **k)
            self._emit()
            return out
    
        def close(self):
            "Close progress bar and emit final callback if needed"
            try:
                super().close()
            finally:
                # only force if we didn't already emit done
                if not self._done_emitted:
                    self._emit(force=True)

    return CallbackTqdm

In [ ]:
#| export
@contextmanager
def patch_tqdm(
    progress_callback: Optional[Callable[[ProgressInfo], None]],  # Function to call with progress updates
    min_update_interval: float = 0.1,  # Minimum time between callback invocations (seconds)
    min_delta_pct: float = 10.0,   # e.g., only every ~10%
    emit_initial: bool = False  # Whether to emit callback at 0% progress
):
    "Context manager that patches tqdm to emit progress callbacks"
    modules = ['tqdm', 'tqdm.std', 'tqdm.auto', 'tqdm.notebook', 'tqdm.asyncio', 'tqdm.gui']
    originals, patched = {}, []
    try:
        for name in modules:
            try:
                mod = import_module(name)
                if hasattr(mod, 'tqdm'):
                    originals[name] = mod.tqdm
                    mod.tqdm = _make_callback_class(
                        mod.tqdm, progress_callback,
                        min_update_interval=min_update_interval,
                        min_delta_pct=min_delta_pct,
                        emit_initial=emit_initial
                    )
                    patched.append(name)
            except Exception:
                pass
        yield
    finally:
        for name in patched:
            try:
                import_module(name).tqdm = originals[name]
            except Exception:
                pass

In [ ]:
def on_progress(info: ProgressInfo):
    print(f"{info.description}: {info.progress:.1f}% ({info.current}/{info.total})")

In [ ]:
#| eval: false
def third_party_function():
    # Simulating a third-party library function that uses tqdm
    from tqdm import tqdm
    import time
    
    results = []
    for i in tqdm(range(100), desc="Downloading"):
        time.sleep(0.01)
        results.append(i * 2)
    return results

with patch_tqdm(on_progress, min_delta_pct=10, emit_initial=False):
    result = third_party_function()
    print(result)

Downloading:  20%|███████████                                            | 20/100 [00:00<00:00, 99.05it/s]

Downloading: 10.0% (10/100)
Downloading: 20.0% (20/100)


Downloading:  40%|██████████████████████                                 | 40/100 [00:00<00:00, 99.01it/s]

Downloading: 30.0% (30/100)
Downloading: 40.0% (40/100)


Downloading:  60%|█████████████████████████████████                      | 60/100 [00:00<00:00, 99.02it/s]

Downloading: 50.0% (50/100)
Downloading: 60.0% (60/100)


Downloading:  80%|████████████████████████████████████████████           | 80/100 [00:00<00:00, 99.01it/s]

Downloading: 70.0% (70/100)
Downloading: 80.0% (80/100)


Downloading: 100%|██████████████████████████████████████████████████████| 100/100 [00:01<00:00, 98.96it/s]

Downloading: 90.0% (90/100)
Downloading: 100.0% (100/100)
[0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40, 42, 44, 46, 48, 50, 52, 54, 56, 58, 60, 62, 64, 66, 68, 70, 72, 74, 76, 78, 80, 82, 84, 86, 88, 90, 92, 94, 96, 98, 100, 102, 104, 106, 108, 110, 112, 114, 116, 118, 120, 122, 124, 126, 128, 130, 132, 134, 136, 138, 140, 142, 144, 146, 148, 150, 152, 154, 156, 158, 160, 162, 164, 166, 168, 170, 172, 174, 176, 178, 180, 182, 184, 186, 188, 190, 192, 194, 196, 198]


In [ ]:
#| eval: false
with patch_tqdm(on_progress, min_delta_pct=10, emit_initial=False):
    from tqdm import tqdm
    for _ in tqdm(range(100), desc="Processing"):
        time.sleep(0.01)

Processing:  20%|███████████▏                                            | 20/100 [00:00<00:00, 99.17it/s]

Processing: 10.0% (10/100)
Processing: 20.0% (20/100)


Processing:  40%|██████████████████████▍                                 | 40/100 [00:00<00:00, 98.86it/s]

Processing: 30.0% (30/100)
Processing: 40.0% (40/100)


Processing:  60%|█████████████████████████████████▌                      | 60/100 [00:00<00:00, 98.96it/s]

Processing: 50.0% (50/100)
Processing: 60.0% (60/100)


Processing:  80%|████████████████████████████████████████████▊           | 80/100 [00:00<00:00, 98.98it/s]

Processing: 70.0% (70/100)
Processing: 80.0% (80/100)


Processing: 100%|███████████████████████████████████████████████████████| 100/100 [00:01<00:00, 98.95it/s]

Processing: 90.0% (90/100)
Processing: 100.0% (100/100)


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()